# Лабораторная работа №3: Классификация. Деревья решений

Автор: Федорова Софья Александровна, 6401-010302D

Датасет Pre-owner cars предоставляет исчерпывающую информацию о подержанных автомобилях, доступных в Индии. Он предлагает ценные инсайты для исследователей, аналитиков и компаний, работающих в автомобильной отрасли, особенно для тех, кто интересуется тенденциями рынка подержанных автомобилей, ценами и предпочтениями клиентов. Целевой признак - цена автомобиля.

Ссылка на датасет: https://www.kaggle.com/datasets/mrmars1010/cars-india-pre-owned

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sbs
import scipy.stats
import  plotly
from matplotlib.ticker import ScalarFormatter

## 1. Считать в `pandas.DataFrame` любой источник данных: CSV, JSON, Excel-файл, HTML-таблицу и т.п.

### Считывание датасета из csv-файла

In [3]:
file = 'pre-owned_cars.csv'

df = pd.read_csv(file)

## 2.  Датасет и подготовка данных:

#### Привести описание датасета.


  
   | Признак | Тип признака | Описание | Единицы измерения |   
   | --- | --- | --- | --- | 
   | brand | categorical | Марка или производитель автомобиля |  
   | model | categorical | Конкретная модель автомобиля| 
   | transmission | categorical | Тип трансмиссии | **целевой признак в задачах бинарной классификации**
   | make_year | categorical |Год производства автомобиля | 
   | reg_year | categorical | Год регистрации автомобиля | 
   | fuel_type  | categorical | Тип топлива, используемого автомобилем | **целевой признак в задачах многоклассовой классификации**
   | engine_capacity(CC) | quantitative | Объём двигателя в кубических сантиметрах | sm**3
   | km_driven | quantitative | Общее расстояние, которое автомобиль прошёл | km
   | ownership | categorical | Количество предыдущих владельцев автомобиля | 
   | **price** | **quantitative** | **Запрашиваемая цена за машину** | какая-то валюта (целевой признак в задаче регрессии)
   | overall_cost | quantitative  | Издержки |
   | has_insurance | categorical | Наличие страховки на машину |
   | spare_key | categorical | Наличие запасного ключа | 
   | reg_number | categorical | Регистрационный номер | 
   | title  | categorical | Краткая информация о машине (марка, год выпуска, бренд) |



#### Осуществить предобработку данных (избавиться от null, убрать некоторые признаки и т.п.) – "подчистить данные".

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2806 entries, 0 to 2805
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   brand                2805 non-null   object 
 1   model                2805 non-null   object 
 2   transmission         2805 non-null   object 
 3   make_year            2805 non-null   float64
 4   reg_year             720 non-null    object 
 5   fuel_type            2805 non-null   object 
 6   engine_capacity(CC)  2688 non-null   float64
 7   km_driven            2805 non-null   float64
 8   ownership            2805 non-null   object 
 9   price                2806 non-null   int64  
 10  overall_cost         2805 non-null   float64
 11  has_insurance        2805 non-null   object 
 12  spare_key            2805 non-null   object 
 13  reg_number           2805 non-null   object 
 14  title                2805 non-null   object 
dtypes: float64(4), int64(1), object(10)
me

Год регистрации автомобиля `reg_year` коррелирует с годом выпуска, поэтому можно избавиться от этого столбца.

Также столбец `title` дублирует информацию, хранящуюся в таблице (title = make_year + brand + model), поэтому его тоже удалим.

In [5]:
df.drop(columns=['reg_year', 'title'], inplace=True)

In [6]:
df = df.dropna()

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2688 entries, 0 to 2804
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   brand                2688 non-null   object 
 1   model                2688 non-null   object 
 2   transmission         2688 non-null   object 
 3   make_year            2688 non-null   float64
 4   fuel_type            2688 non-null   object 
 5   engine_capacity(CC)  2688 non-null   float64
 6   km_driven            2688 non-null   float64
 7   ownership            2688 non-null   object 
 8   price                2688 non-null   int64  
 9   overall_cost         2688 non-null   float64
 10  has_insurance        2688 non-null   object 
 11  spare_key            2688 non-null   object 
 12  reg_number           2688 non-null   object 
dtypes: float64(4), int64(1), object(8)
memory usage: 294.0+ KB


#### Закодировать категориальные признаки при необходимости.

In [8]:
numeric_cols =  ['price', 'km_driven', 'engine_capacity(CC)', 'make_year', 'overall_cost']
categorial_cols = ['brand', 'model', 'transmission', 'fuel_type', 'ownership', 'has_insurance', 'spare_key', 'reg_number']

df_categorial = df[categorial_cols]
df_categorial['model'].value_counts().sort_index() #категория model содержит 785 уникальных значений
df_categorial['brand'].value_counts().sort_index() #категория brand содержит 15 уникальных значений
df_categorial['transmission'].value_counts().sort_index() #категория transmission содержит 2 уникальных значения
df_categorial['fuel_type'].value_counts().sort_index() #категория transmission содержит 3 уникальных значения

df_categorial['ownership'].value_counts().sort_index() #категория ownership содержит 3 уникальных значения
df_categorial['has_insurance'].value_counts().sort_index() #категория has_insurance содержит только True
df_categorial['spare_key'].value_counts().sort_index() #категория spare_key содержит 2 уникальных значения (No-1962, Yes-726)
df_categorial['reg_number'].value_counts().sort_index(); #категория reg_number содержит 153 уникальных значения

В результате анализа уникальных значений для категориальных признаков, удаляем признак has_insurance, содержащий только значения одного класса. Целевым классом в задаче классификации будет `transmission`.

In [9]:
df = df.drop(columns='has_insurance');

Для разных типов данных будем использовать разные способы кодирования признаков - для признаков с малым количеством категорий применяем `One-Hot Encoding`, для признаков с большим количеством категорий - `Label Encoding`.


При использовании One-Hot Encoding для признаков с большим количеством категорий (brand, model, reg_number) возникло бы увеличение размерности, то есть размерность стала бы содержать 961 признак. Для алгоритма kNN станет проблематично из-за "проклятия размерности" (расстояния в высокоразмерном пространстве становятся менее информативными).
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [10]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()

onehot_cols = ['transmission', 'ownership', 'spare_key']
df_encoded = pd.get_dummies(df_encoded, columns=onehot_cols, drop_first=True)

label_cols = ['brand', 'model', 'reg_number', 'fuel_type']
label_encoders = {} # здесь хранится словарь с моделями для каждого столбца категориального признака

for col in label_cols:
    le = LabelEncoder()
    df_encoded[col + '_encoded'] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le
    df_encoded.drop(col, axis=1, inplace=True)

print(dict(enumerate(label_encoders['fuel_type'].classes_)))

{0: 'CNG', 1: 'Diesel', 2: 'Petrol'}


Нормализовать данные.

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numeric_cols = ['make_year', 'engine_capacity(CC)', 'km_driven', 'price', 'overall_cost']

scaler.fit(df_encoded[numeric_cols])

df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])


Разбить выборку на обучающую и тестовую.

In [12]:
y = df_encoded['fuel_type_encoded']
df_filtered = df_encoded.drop(columns='fuel_type_encoded')
X = df_filtered

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, train_size=0.75, random_state=42
)

При наличии дисбаланса классов снизить дисбаланс классов.

В нашей целевой переменной есть дисбаланс, который будем снижать с помощью метода `SMOTE`. Метод находит ближайших соседей миноритарного класса и генерирует образцы, делая линейную интерполяцию между ними. Обычно образец выбирается случайным образом из малочисленного класса. Это делается для того, чтобы обеспечить разнообразие и предотвратить смещение в сторону какой-либо конкретной области пространства признаков. 

In [14]:
print(y.value_counts(normalize=True))

fuel_type_encoded
2    0.850446
1    0.111607
0    0.037946
Name: proportion, dtype: float64


In [15]:
from imblearn.combine import SMOTETomek

# SMOTE + Tomek links (удаляет "шумные" образцы)
smote_tomek = SMOTETomek(random_state=42)
X_train_resampled, y_train_resampled = smote_tomek.fit_resample(X_train, y_train)

print(pd.Series(y_train_resampled).value_counts().sort_index())

fuel_type_encoded
0    1673
1    1671
2    1633
Name: count, dtype: int64


## 3.  Дерево решений

С использованием `GridSearchCV` осуществить подбор гиперпараметра `DecisionTreeClassifier` (как минимум `max_depth`, `max_features`, другие параметры &ndash; по желанию.)
   - Вывести значения гиперпараметра и метрик для наилучшей модели `DecisionTreeClassifier` ($accuracy$, $precision$, $recall$, $\textit{f-measure}$).

Для каждой уникальной пары значений параметров `max_depth` и `max_features` будет проведена $5$-кратная кросс-валидация и выберется лучшее сочетание параметров.

In [16]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

model_tree_params = {
    'max_depth': range(1, 11),
    'max_features': range(2, 12) #максимальное количество признаков, которые есть
}

# Создаем модель
model_tree = DecisionTreeClassifier(random_state=42)

model_tree_grid = GridSearchCV(
    model_tree, model_tree_params,
    cv = 5, 
    scoring='accuracy', # для меры оценки выбранной пары признаков
    n_jobs = -1,  # используем все ядра процессора
    verbose = True
)

# Обучаем на всех обучающих данных (с автоматическим перебором)
model_tree_grid.fit(X_train_resampled, y_train_resampled);

Fitting 5 folds for each of 100 candidates, totalling 500 fits


Лучшее сочетание параметров и соответствующая средняя доля правильных ответов на кросс-валидации:

In [17]:
model_tree_grid.best_params_

{'max_depth': 10, 'max_features': 11}

Лучшая точность на кросс-валидации

In [18]:
model_tree_grid.best_score_

0.9314869528364715

Вывести значения гиперпараметра и метрик для наилучшей модели `DecisionTreeClassifier` ($accuracy$, $precision$, $recall$, $\textit{f-measure}$).

In [19]:
best_tree = model_tree_grid.best_estimator_
y_pred = best_tree.predict(X_test)

In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy = accuracy_score(y_test, model_tree_grid.predict(X_test))
precision = precision_score(y_test, model_tree_grid.predict(X_test),  average='weighted')
recall = recall_score(y_test, model_tree_grid.predict(X_test),  average='weighted')
f_measure = 2*(precision*recall)/(precision+recall)

print(f'Test accuracy: {accuracy}')
print(f'Test precision: {precision}')
print(f'Test recall: {recall}')
print(f'Test f-measure: {f_measure}')

Test accuracy: 0.9032738095238095
Test precision: 0.9248750078735197
Test recall: 0.9032738095238095
Test f-measure: 0.9139467900481195


Результаты точности для модели с глубиной 'max_depth' = 10 и 'max_features' = 10

In [21]:
# {0: 'CNG', 1: 'Diesel', 2: 'Petrol'}
import sklearn

report = sklearn.metrics.classification_report(y_test, y_pred, target_names=["CNG", "Diesel", 'Petrol'], zero_division=0)
print(report)

              precision    recall  f1-score   support

         CNG       0.40      0.73      0.51        26
      Diesel       0.76      0.85      0.81        75
      Petrol       0.97      0.92      0.94       571

    accuracy                           0.90       672
   macro avg       0.71      0.83      0.75       672
weighted avg       0.92      0.90      0.91       672



#### Важность признаков лучшей модели

Для полученного наилучшего дерева вывести `feature_importances`, отсортировать их по убыванию.

In [22]:
# zip объединяет элементы из нескольких итерируемых объектов в кортежи
# model_tree_grid.best_estimator_.feature_importances_ - хранит числовые значения весов признаков
colums = X_train_resampled.columns
features = zip(X_train_resampled.columns, model_tree_grid.best_estimator_.feature_importances_)
features = sorted(features, key=lambda x: x[1], reverse=True)
len_max = max([len(col) for col in X_train_resampled.columns])
for f_name, f_val in features:
    print(f"{f_name:<{len_max}} importance: {f_val:.5f}")

engine_capacity(CC) importance: 0.44139
model_encoded       importance: 0.08526
transmission_Manual importance: 0.08001
overall_cost        importance: 0.07727
make_year           importance: 0.07423
price               importance: 0.07387
brand_encoded       importance: 0.05694
km_driven           importance: 0.05284
reg_number_encoded  importance: 0.04193
ownership_2nd owner importance: 0.00868
ownership_3rd owner importance: 0.00618
spare_key_Yes       importance: 0.00141


Осуществить фильтрацию признаков (по какому-нибудь значению порога важности признака или другим способом)

Отсортируем признаки по медианному значению

In [23]:
import numpy as np

f_val_t = np.median([v for f, v in features])
f_filtered = [f for f, v in features if v > f_val_t]
f_filtered

['engine_capacity(CC)',
 'model_encoded',
 'transmission_Manual',
 'overall_cost',
 'make_year',
 'price']

Отсортируем признаки, используя метод исключений (leave-one-out)

In [24]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt

def leave_one_out_feature_importance(X, y, model, metric=accuracy_score, random_state=42):

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=random_state, stratify=y
    )
    
    if isinstance(model, type):
        # Если передан класс модели
        base_model = model(random_state=random_state)
    else:
        # Если передана обученная модель
        base_model = model.__class__(**model.get_params())
    
    base_model.fit(X_train, y_train)
    base_score = metric(y_test, base_model.predict(X_test))
    
    print(f"Значение точности для всех признаков: {base_score:.4f}")

    importance = {}
 
    for i, feature in enumerate(X.columns):
        X_train_reduced = X_train.drop(columns=[feature])
        X_test_reduced = X_test.drop(columns=[feature])
        
        # Обучаем модель без признака
        if isinstance(model, type):
            reduced_model = model(random_state=random_state)
        else:
            reduced_model = model.__class__(**model.get_params())
        
        reduced_model.fit(X_train_reduced, y_train)
        reduced_score = metric(y_test, reduced_model.predict(X_test_reduced))
        
        # Изменение точности
        importance[feature] = base_score - reduced_score
        
        print(f"Удален: {feature:<20} | Точность: {reduced_score:.4f} | "
              f"Метрика {'ухудшилась' if base_score > reduced_score else 'улучшилась'}")
    
    importance_df = pd.DataFrame({
        'feature': list(importance.keys()),
        'importance': list(importance.values())
    }).sort_values('importance', ascending=False)
    
    return importance_df

importance_df = leave_one_out_feature_importance(
    X_train_resampled, 
    y_train_resampled,
    DecisionTreeClassifier(max_depth=10, random_state=42, max_features=10), # лучшая модель
    metric=accuracy_score
)


Значение точности для всех признаков: 0.9257
Удален: make_year            | Точность: 0.9237 | Метрика ухудшилась
Удален: engine_capacity(CC)  | Точность: 0.8896 | Метрика ухудшилась
Удален: km_driven            | Точность: 0.9257 | Метрика улучшилась
Удален: price                | Точность: 0.9284 | Метрика улучшилась
Удален: overall_cost         | Точность: 0.9197 | Метрика ухудшилась
Удален: transmission_Manual  | Точность: 0.9398 | Метрика улучшилась
Удален: ownership_2nd owner  | Точность: 0.9378 | Метрика улучшилась
Удален: ownership_3rd owner  | Точность: 0.9511 | Метрика улучшилась
Удален: spare_key_Yes        | Точность: 0.9384 | Метрика улучшилась
Удален: brand_encoded        | Точность: 0.9418 | Метрика улучшилась
Удален: model_encoded        | Точность: 0.9277 | Метрика улучшилась
Удален: reg_number_encoded   | Точность: 0.9458 | Метрика улучшилась


In [25]:
f_leave_one = list(importance_df['feature'].head(6))

In [26]:
f_filtered

['engine_capacity(CC)',
 'model_encoded',
 'transmission_Manual',
 'overall_cost',
 'make_year',
 'price']

В результате сравнения двух подходов, в список важных признаков добавлены следующие столбцы `engine_capacity(CC)`, `model_encoded`, `km_driven`, `make_year`, `overall_cost`.

In [27]:
importance_features = ['engine_capacity(CC)', 'model_encoded', 'km_driven', 'make_year', 'overall_cost']

Из прошлой лабораторной работы (результаты получены в результате анализа матрицы корреляции): 


Наша целевая переменная имеет выраженную корреляцию со следующими переменными `engine_capacity(CC)`, `km_driven`, `price`, `overall_cost`, `transmission_Manual`, `model_encoded`, именно их мы оставим.

In [28]:
correlated_columns = ['engine_capacity(CC)', 'km_driven', 'price', 'overall_cost', 'transmission_Manual', 'model_encoded']

Подобрать лучшую модель с использованием `GridSearchCV` на обучающей выборке с отфильтрованными признаками. Здесь я решила сравнить сразу несколько списков важных признаков, полученных разными способами.

In [29]:
from sklearn.model_selection import GridSearchCV

feature_sets = {
    'filtered': f_filtered,  # список признаков после медианной фильтрации
    'leave_one': f_leave_one,  # список после leave-one-out
    'importance': importance_features, # гибрид двух списков (общее)
    'correlated_matrix': correlated_columns # список, полученный из анализа корреляционной матрицы
}

model_tree_params = {
    'max_depth': range(1, 11),
    'max_features': range(2, 12) # максимальное количество признаков, которые есть
}

results = {}

for name, features in feature_sets.items():

    X_train_selected = X_train_resampled[features]
    
    grid_search = GridSearchCV(
        DecisionTreeClassifier(random_state=42),
        model_tree_params,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train_selected, y_train_resampled)
    
    X_test_selected = X_test[features]
    y_pred = grid_search.predict(X_test_selected)
    
    results[name] = {
        'len(features)': features,
        'best_model': grid_search.best_estimator_,
        'best_params': grid_search.best_params_,
        'best_cv_score': grid_search.best_score_,
        'test_accuracy': accuracy_score(y_test, y_pred),
        'n_features': len(features),
        'features': features
    }

    print(f"\nСписок входных признаков: {features}")
    print(f"Лучшие параметры: {grid_search.best_params_}")
    print(f"Лучшая CV точность: {grid_search.best_score_:.4f}")
    print(f"Тестовая точность: {results[name]['test_accuracy']:.4f} \n")


Fitting 5 folds for each of 100 candidates, totalling 500 fits

Список входных признаков: ['engine_capacity(CC)', 'model_encoded', 'transmission_Manual', 'overall_cost', 'make_year', 'price']
Лучшие параметры: {'max_depth': 10, 'max_features': 4}
Лучшая CV точность: 0.9321
Тестовая точность: 0.9048 

Fitting 5 folds for each of 100 candidates, totalling 500 fits

Список входных признаков: ['engine_capacity(CC)', 'overall_cost', 'make_year', 'km_driven', 'model_encoded', 'price']
Лучшие параметры: {'max_depth': 10, 'max_features': 6}
Лучшая CV точность: 0.9311
Тестовая точность: 0.8467 

Fitting 5 folds for each of 100 candidates, totalling 500 fits

Список входных признаков: ['engine_capacity(CC)', 'model_encoded', 'km_driven', 'make_year', 'overall_cost']
Лучшие параметры: {'max_depth': 10, 'max_features': 5}
Лучшая CV точность: 0.9279
Тестовая точность: 0.8601 

Fitting 5 folds for each of 100 candidates, totalling 500 fits

Список входных признаков: ['engine_capacity(CC)', 'km_drive

### Наилучшие результаты по точности получены в результаты использования признаков по `медианной фильтрации`.

In [30]:
model_tree_params = {
    'max_depth': range(1, 11),
    'max_features': range(2, 12) #максимальное количество признаков, которые есть
}

model_tree_grid_filtered = GridSearchCV(
    model_tree, model_tree_params,
    cv = 5, n_jobs = -1,
    verbose = True
)

model_tree_grid_filtered.fit(X_train_resampled[f_filtered], y_train_resampled);

Fitting 5 folds for each of 100 candidates, totalling 500 fits


Вывести полученные гиперпараметры лучшей модели.

In [31]:
model_tree_grid_filtered.best_params_

{'max_depth': 10, 'max_features': 4}

Лучшая точность на кросс-валидации

In [32]:
model_tree_grid_filtered.best_score_

0.9320893624750257

Обучение модели

In [33]:
best_filtered_tree = model_tree_grid_filtered.best_estimator_
y_filtered_pred = model_tree_grid_filtered.predict(X_test[f_filtered])

In [34]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy_filtered = accuracy_score(y_test, model_tree_grid_filtered.predict(X_test[f_filtered]))
precision_filtered = precision_score(y_test, model_tree_grid_filtered.predict(X_test[f_filtered]),  average='weighted')
recall_filtered = recall_score(y_test, model_tree_grid_filtered.predict(X_test[f_filtered]),  average='weighted')
f_measure_filtered = 2*(precision_filtered*recall_filtered)/(precision_filtered+recall_filtered)

print(f'Test accuracy: {accuracy_filtered}')
print(f'Test precision: {precision_filtered}')
print(f'Test recall: {recall_filtered}')
print(f'Test f-measure: {f_measure_filtered}')

Test accuracy: 0.9047619047619048
Test precision: 0.9297564564884275
Test recall: 0.9047619047619048
Test f-measure: 0.9170889104253127


In [35]:
print(f"Before:")
print(f"  input:        {len(colums)} features")
print(f"  params:       {model_tree_grid.best_params_}")
print(f"  train score:  {model_tree_grid.best_score_}")
print(f"  test score:   {accuracy_score(y_test, model_tree_grid.predict(X_test))}")
print(f"After:")
print(f"  input:        {len(f_filtered)} features")
print(f"  params:       {model_tree_grid_filtered.best_params_}")
print(f"  train score:  {model_tree_grid_filtered.best_score_}")
print(f"  test score:   {accuracy_score(y_test, model_tree_grid_filtered.predict(X_test[f_filtered]))}")

Before:
  input:        12 features
  params:       {'max_depth': 10, 'max_features': 11}
  train score:  0.9314869528364715
  test score:   0.9032738095238095
After:
  input:        6 features
  params:       {'max_depth': 10, 'max_features': 4}
  train score:  0.9320893624750257
  test score:   0.9047619047619048


Сокращение количества признаков немного улучшило метрики (на 0,01) для модели. Этот эксперимент показал, как важно правильно выбрать признаки, потому что на некоторых разбиениях (corr matrix или leave-one-out) значения точности для тестовой выборки значительно снизилось.



## 4.  Случайный лес

Построить случайный лес (`RandomForestClassifier`), c использованием `GridSearchCV` осуществить подбор гиперпараметра.

In [36]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

model_forest_params = {
    'max_depth': range(1, 11),
    'max_features': range(2, 12) #максимальное количество признаков, которые есть
}

# Создаем модель
model_forest = RandomForestClassifier(random_state=42)

model_forest_grid = GridSearchCV(
    model_forest, model_forest_params,
    cv = 5, 
    scoring='accuracy', # для меры оценки выбранной пары признаков
    n_jobs = -1,  # используем все ядра процессора
    verbose = True
)

# Обучаем на всех обучающих данных (с автоматическим перебором)
model_forest_grid.fit(X_train_resampled, y_train_resampled);

Fitting 5 folds for each of 100 candidates, totalling 500 fits


Вывести полученные гиперпараметры лучшей модели случайного леса.

In [37]:
model_forest_grid.best_params_

{'max_depth': 10, 'max_features': 5}

In [38]:
accuracy_score(y_test, model_forest_grid.predict(X_test))

0.9151785714285714

Результаты обучения лучшей модели на исходном наборе признаков.

In [39]:
best_forest = model_forest_grid.best_estimator_
y_pred = model_forest_grid.predict(X_test)

Осуществить фильтрацию признаков.

Фильтрацию признаков осуществим, учитывая `Permutation Importance` (Важность перестановок). 

"Модельно-независимый" способ:
- Измеряется базовая точность модели на тестовом наборе (метрика по выбору).
- Значения одного признака случайным образом перемешиваются (разрушая связь с целевой переменной).
- Разница в точности до и после перемешивания считается мерой важности этого признака.

In [40]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def compute_permutation_importance_sklearn(model, X_test, y_test, metric='accuracy', n_repeats=10):

    result = permutation_importance(
        model, X_test, y_test,
        n_repeats=n_repeats,
        random_state=42,
        scoring=metric  
    )

    importance_df = pd.DataFrame({
        'feature': X_test.columns,
        'importance': result.importances_mean,
        'std': result.importances_std
    }).sort_values('importance', ascending=False)
    
    return importance_df, result

perm_importance_sklearn, result = compute_permutation_importance_sklearn(
    model_forest_grid, X_test, y_test,
    metric='accuracy',
    n_repeats=10
)

print("Permutation Importance (sklearn):")
print(perm_importance_sklearn.head(10))

Permutation Importance (sklearn):
                feature  importance       std
1   engine_capacity(CC)    0.155655  0.008370
10        model_encoded    0.040476  0.008036
4          overall_cost    0.035417  0.006942
5   transmission_Manual    0.028869  0.005771
3                 price    0.018155  0.007068
0             make_year    0.016815  0.004938
9         brand_encoded    0.015625  0.003660
2             km_driven    0.013095  0.005060
11   reg_number_encoded    0.010863  0.004417
8         spare_key_Yes    0.002381  0.001906


Выделим признаки, найденные этим способом в особую переменную для проверки.

In [41]:
perm_importance_sklearn_features = list(perm_importance_sklearn['feature'].head(6))

perm_importance_sklearn_features

['engine_capacity(CC)',
 'model_encoded',
 'overall_cost',
 'transmission_Manual',
 'price',
 'make_year']

Подобрать лучшую модель с использованием `GridSearchCV` на обучающей выборке с отфильтрованными признаками.

In [42]:
from sklearn.model_selection import GridSearchCV

feature_sets = {
    'filtered': f_filtered,  # список признаков после медианной фильтрации
    'leave_one': f_leave_one,  # список после leave-one-out
    'importance': importance_features, # гибрид двух списков (общее)
    'correlated_matrix': correlated_columns, # список, полученный из анализа корреляционной матрицы
    'perm_importance': perm_importance_sklearn_features # список, полученный в результате учета важности перестановок
}

model_forest_params = {
    'max_depth': range(1, 11),
    'max_features': range(2, 12) # максимальное количество признаков, которые есть
}

results = {}

for name, features in feature_sets.items():

    X_train_selected = X_train_resampled[features]
    
    grid_search = GridSearchCV(
        RandomForestClassifier(random_state=42),
        model_tree_params,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train_selected, y_train_resampled)
    
    X_test_selected = X_test[features]
    y_pred = grid_search.predict(X_test_selected)
    
    results[name] = {
        'len(features)': features,
        'best_model': grid_search.best_estimator_,
        'best_params': grid_search.best_params_,
        'best_cv_score': grid_search.best_score_,
        'test_accuracy': accuracy_score(y_test, y_pred),
        'n_features': len(features),
        'features': features
    }

    print(f"\nСписок входных признаков: {features}")
    print(f"Лучшие параметры: {grid_search.best_params_}")
    print(f"Лучшая CV точность: {grid_search.best_score_:.4f}")
    print(f"Тестовая точность: {results[name]['test_accuracy']:.4f} \n")


Fitting 5 folds for each of 100 candidates, totalling 500 fits

Список входных признаков: ['engine_capacity(CC)', 'model_encoded', 'transmission_Manual', 'overall_cost', 'make_year', 'price']
Лучшие параметры: {'max_depth': 10, 'max_features': 4}
Лучшая CV точность: 0.9588
Тестовая точность: 0.9018 

Fitting 5 folds for each of 100 candidates, totalling 500 fits

Список входных признаков: ['engine_capacity(CC)', 'overall_cost', 'make_year', 'km_driven', 'model_encoded', 'price']
Лучшие параметры: {'max_depth': 10, 'max_features': 3}
Лучшая CV точность: 0.9596
Тестовая точность: 0.8973 

Fitting 5 folds for each of 100 candidates, totalling 500 fits

Список входных признаков: ['engine_capacity(CC)', 'model_encoded', 'km_driven', 'make_year', 'overall_cost']
Лучшие параметры: {'max_depth': 10, 'max_features': 2}
Лучшая CV точность: 0.9536
Тестовая точность: 0.9003 

Fitting 5 folds for each of 100 candidates, totalling 500 fits

Список входных признаков: ['engine_capacity(CC)', 'km_drive

Наилучшие значения точности на тренировочных и тестовых данных получены для признаков, которые отфильторованы методом `Permutation Importance`, именно на такой тренировочной выборке будем строить наилучший алгоритм.

In [43]:
model_forest_params = {
    'max_depth': range(1, 11),
    'max_features': range(2, 12) #максимальное количество признаков, которые есть
}

model_forest_grid_filtered = GridSearchCV(
    model_forest, model_forest_params,
    cv = 5, n_jobs = -1,
    verbose = True
)

model_forest_grid_filtered.fit(X_train_resampled[perm_importance_sklearn_features], y_train_resampled);

Fitting 5 folds for each of 100 candidates, totalling 500 fits


Вывести полученные гиперпараметры лучшей модели случайного леса.

In [44]:
model_forest_grid_filtered.best_params_

{'max_depth': 10, 'max_features': 4}

In [45]:
model_forest_grid_filtered.best_score_

0.9594161570906742

In [46]:
best_filtered_forest = model_forest_grid_filtered.best_estimator_
y_filtered_pred = model_forest_grid_filtered.predict(X_test[perm_importance_sklearn_features])

Сравнить метрики до и после фильтрации признаков лучших моделей.

In [47]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy_filtered = accuracy_score(y_test, model_forest_grid_filtered.predict(X_test[perm_importance_sklearn_features]))
precision_filtered = precision_score(y_test, model_forest_grid_filtered.predict(X_test[perm_importance_sklearn_features]),  average='weighted')
recall_filtered = recall_score(y_test, model_forest_grid_filtered.predict(X_test[perm_importance_sklearn_features]),  average='weighted')
f_measure_filtered = 2*(precision_filtered*recall_filtered)/(precision_filtered+recall_filtered)

print(f'Test accuracy: {accuracy_filtered}')
print(f'Test precision: {precision_filtered}')
print(f'Test recall: {recall_filtered}')
print(f'Test f-measure: {f_measure_filtered}')
print(f"Before:")
print(f"  input:        {len(colums)} features")
print(f"  params:       {model_forest_grid.best_params_}")
print(f"  train score:  {model_forest_grid.best_score_}")
print(f"  test score:   {accuracy_score(y_test, model_forest_grid.predict(X_test))}")
print(f"After:")
print(f"  input:        {len(perm_importance_sklearn_features)} features")
print(f"  params:       {model_forest_grid_filtered.best_params_}")
print(f"  train score:  {model_forest_grid_filtered.best_score_}")
print(f"  test score:   {accuracy_score(y_test, model_forest_grid_filtered.predict(X_test[perm_importance_sklearn_features]))}")

Test accuracy: 0.90625
Test precision: 0.9322561351752177
Test recall: 0.90625
Test f-measure: 0.9190691359014936
Before:
  input:        12 features
  params:       {'max_depth': 10, 'max_features': 5}
  train score:  0.9672525276987345
  test score:   0.9151785714285714
After:
  input:        6 features
  params:       {'max_depth': 10, 'max_features': 4}
  train score:  0.9594161570906742
  test score:   0.90625


В результата уменьшения количества признаков точность снизилась на 0,01 относительно начальных результатов.

## 5. Метод ближайших соседей

С использованием `GridSearchCV` осуществить подбор гиперпараметра `KNeighborsClassifier` (`n_neighbors`).

In [48]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

model_knn_params = {
    'n_neighbors': range(2, 16)
}

model_knn = KNeighborsClassifier()

model_knn_grid = GridSearchCV(
    model_knn, model_knn_params,
    cv = 5, 
    scoring='accuracy', # для меры оценки выбранной пары признаков
    n_jobs = -1,  # используем все ядра процессора
    verbose = True
)

# Обучаем на всех обучающих данных (с автоматическим перебором)
model_knn_grid.fit(X_train_resampled, y_train_resampled);

Fitting 5 folds for each of 14 candidates, totalling 70 fits


In [49]:
y_pred = model_knn_grid.predict(X_test)

Вывести значения гиперпараметра и метрик для наилучшей модели.

In [50]:
model_knn_grid.best_params_

{'n_neighbors': 3}

In [51]:
model_knn_grid.best_score_

0.8734199107989747

Осуществить фильтрацию признаков.

Для предварительного отбора признаков ранее использовались модели (деревья решений, случайный лес). Важность, полученная от них, используется для настройки kNN. Комбинирование с `RFE` (recursive feature elimination) позволит автоматически отобрать наиболее важные признаки.

In [52]:
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance
from collections import Counter
from scipy.stats import spearmanr


def ensemble_feature_selection(X_train, y_train, X_test, y_test, 
                               n_final_features=6):
  
    
    estimator = DecisionTreeClassifier(max_depth=5, random_state=42)
    rfe = RFE(estimator, n_features_to_select=n_final_features, step=1)
    rfe.fit(X_train, y_train)
    rfe_features = X_train.columns[rfe.support_].tolist()
    print(f"   RFE выбрал: {len(rfe_features)} признаков")
    
    model = DecisionTreeClassifier(max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    
    perm_importance = permutation_importance(
        model, X_test, y_test,
        n_repeats=10, random_state=42, scoring='accuracy'
    )
    
    perm_features = X_train.columns[
        np.argsort(perm_importance.importances_mean)[-n_final_features:]
    ].tolist()
    print(f"   Permutation Importance выбрал: {len(perm_features)} признаков")
    
    tree_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    tree_features = tree_importance.head(n_final_features)['feature'].tolist()
    print(f"   Feature Importance выбрал: {len(tree_features)} признаков")
    
    correlations = []
    for feature in X_train.columns:
        corr, _ = spearmanr(X_train[feature], y_train)
        correlations.append(abs(corr))
    
    corr_df = pd.DataFrame({
        'feature': X_train.columns,
        'correlation': correlations
    }).sort_values('correlation', ascending=False)
    
    corr_features = corr_df.head(n_final_features)['feature'].tolist()
    print(f"   Корреляция выбрала: {len(corr_features)} признаков")
    
    all_features = rfe_features + perm_features + tree_features + corr_features
    
    # Подсчитываем голоса
    feature_votes = Counter(all_features)
    
    # Сортируем по количеству голосов
    feature_votes_df = pd.DataFrame(
        feature_votes.items(), 
        columns=['feature', 'votes']
    ).sort_values('votes', ascending=False)
    
    # Выбираем финальные признаки (те, что получили больше всего голосов)
    final_features = feature_votes_df.head(n_final_features)['feature'].tolist()
    
 
    for i, feature in enumerate(final_features, 1):
        votes = feature_votes[feature]
        print(f"{i:2}. {feature:<30} (голосов: {votes})")
    
    # Обучаем финальную модель
    X_train_final = X_train[final_features]
    X_test_final = X_test[final_features]
    
    final_model = DecisionTreeClassifier(max_depth=5, random_state=42)
    final_model.fit(X_train_final, y_train)
    y_pred = final_model.predict(X_test_final)
    
    accuracy = accuracy_score(y_test, y_pred)

    print(f"Количество признаков: {len(final_features)}")
    print(f"Accuracy: {accuracy:.4f}")
    
    return {
        'selected_features': final_features,
        'feature_votes': feature_votes_df,
        'model': final_model,
        'accuracy': accuracy,
        'individual_results': {
            'rfe': rfe_features,
            'permutation': perm_features,
            'tree_importance': tree_features,
            'correlation': corr_features
        }
    }

# Использование
ensemble_results = ensemble_feature_selection(
    X_train_resampled, y_train_resampled,
    X_test, y_test,
    n_final_features=6
)

   RFE выбрал: 6 признаков
   Permutation Importance выбрал: 6 признаков
   Feature Importance выбрал: 6 признаков
   Корреляция выбрала: 6 признаков
 1. engine_capacity(CC)            (голосов: 4)
 2. price                          (голосов: 4)
 3. transmission_Manual            (голосов: 4)
 4. model_encoded                  (голосов: 4)
 5. make_year                      (голосов: 3)
 6. overall_cost                   (голосов: 3)
Количество признаков: 6
Accuracy: 0.7277


In [53]:
esemble_features = list(ensemble_results['feature_votes']['feature'])

esemble_features

['engine_capacity(CC)',
 'price',
 'transmission_Manual',
 'model_encoded',
 'make_year',
 'overall_cost',
 'spare_key_Yes',
 'ownership_2nd owner']

Подобрать лучшую модель с использованием `GridSearchCV` на обучающей выборке с отфильтрованными признаками.

In [54]:
from sklearn.model_selection import GridSearchCV

feature_sets = {
    'filtered': f_filtered,  # список признаков после медианной фильтрации
    'leave_one': f_leave_one,  # список после leave-one-out
    'importance': importance_features, # гибрид двух списков (общее)
    'correlated_matrix': correlated_columns, # список, полученный из анализа корреляционной матрицы
    'perm_importance': perm_importance_sklearn_features, # список, полученный в результате учета важности перестановок
    'esemble': esemble_features # список, полученный в результате голосования
}

model_knn_params = {
    'n_neighbors': range(2, 16)
}

results = {}

for name, features in feature_sets.items():

    X_train_selected = X_train_resampled[features]
    
    grid_search = GridSearchCV(
        KNeighborsClassifier(),
        model_knn_params,
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train_selected, y_train_resampled)
    
    X_test_selected = X_test[features]
    y_pred = grid_search.predict(X_test_selected)
    
    results[name] = {
        'len(features)': features,
        'best_model': grid_search.best_estimator_,
        'best_params': grid_search.best_params_,
        'best_cv_score': grid_search.best_score_,
        'test_accuracy': accuracy_score(y_test, y_pred),
        'n_features': len(features),
        'features': features
    }

    print(f"\nСписок входных признаков: {features}")
    print(f"Лучшие параметры: {grid_search.best_params_}")
    print(f"Лучшая CV точность: {grid_search.best_score_:.4f}")
    print(f"Тестовая точность: {results[name]['test_accuracy']:.4f} \n")


Fitting 5 folds for each of 14 candidates, totalling 70 fits

Список входных признаков: ['engine_capacity(CC)', 'model_encoded', 'transmission_Manual', 'overall_cost', 'make_year', 'price']
Лучшие параметры: {'n_neighbors': 2}
Лучшая CV точность: 0.8915
Тестовая точность: 0.8378 

Fitting 5 folds for each of 14 candidates, totalling 70 fits

Список входных признаков: ['engine_capacity(CC)', 'overall_cost', 'make_year', 'km_driven', 'model_encoded', 'price']
Лучшие параметры: {'n_neighbors': 2}
Лучшая CV точность: 0.8782
Тестовая точность: 0.7798 

Fitting 5 folds for each of 14 candidates, totalling 70 fits

Список входных признаков: ['engine_capacity(CC)', 'model_encoded', 'km_driven', 'make_year', 'overall_cost']
Лучшие параметры: {'n_neighbors': 2}
Лучшая CV точность: 0.8718
Тестовая точность: 0.7783 

Fitting 5 folds for each of 14 candidates, totalling 70 fits

Список входных признаков: ['engine_capacity(CC)', 'km_driven', 'price', 'overall_cost', 'transmission_Manual', 'model_enc

Наилучшие значения для точности на тестовой выборке достигаются при выборе параметров, выбранными "вручную" исходя из анализа корреляционной матрицы (`correlated_columns`).

In [55]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

model_knn_params = {
    'n_neighbors': range(2, 16)
}

model_filtered_knn = KNeighborsClassifier()

model_knn_filtered_grid = GridSearchCV(
    model_filtered_knn, model_knn_params,
    cv = 5, 
    scoring='accuracy', # для меры оценки выбранной пары признаков
    n_jobs = -1,  # используем все ядра процессора
    verbose = True
)

model_knn_filtered_grid.fit(X_train_resampled[correlated_columns], y_train_resampled);

Fitting 5 folds for each of 14 candidates, totalling 70 fits


Вывести полученные гиперпараметры лучшей модели случайного леса.

In [56]:
model_knn_filtered_grid.best_params_

{'n_neighbors': 3}

In [57]:
model_knn_filtered_grid.best_score_

0.8722118625254789

Сравнить метрики до и после фильтрации признаков.

In [58]:
y_filtered_pred = model_knn_filtered_grid.predict(X_test[correlated_columns])

In [59]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy_filtered = accuracy_score(y_test, model_knn_filtered_grid.predict(X_test[correlated_columns]))
precision_filtered = precision_score(y_test, model_knn_filtered_grid.predict(X_test[correlated_columns]),  average='weighted')
recall_filtered = recall_score(y_test, model_knn_filtered_grid.predict(X_test[correlated_columns]),  average='weighted')
f_measure_filtered = 2*(precision_filtered*recall_filtered)/(precision_filtered+recall_filtered)

print(f'Test accuracy: {accuracy_filtered}')
print(f'Test precision: {precision_filtered}')
print(f'Test recall: {recall_filtered}')
print(f'Test f-measure: {f_measure_filtered}')
print(f"Before:")
print(f"  input:        {len(colums)} features")
print(f"  params:       {model_knn_grid.best_params_}")
print(f"  train score:  {model_knn_grid.best_score_}")
print(f"  test score:   {accuracy_score(y_test, model_knn_grid.predict(X_test))}")
print(f"After:")
print(f"  input:        {len(correlated_columns)} features")
print(f"  params:       {model_knn_filtered_grid.best_params_}")
print(f"  train score:  {model_knn_filtered_grid.best_score_}")
print(f"  test score:   {accuracy_score(y_test, model_knn_filtered_grid.predict(X_test[correlated_columns]))}")

Test accuracy: 0.8616071428571429
Test precision: 0.9153820412673842
Test recall: 0.8616071428571429
Test f-measure: 0.8876809293442047
Before:
  input:        12 features
  params:       {'n_neighbors': 3}
  train score:  0.8734199107989747
  test score:   0.7410714285714286
After:
  input:        6 features
  params:       {'n_neighbors': 3}
  train score:  0.8722118625254789
  test score:   0.8616071428571429


Фильтрация признаков (сокращение их количества в два раза) для модели kNN улучшила результаты на тестовой выборке на 10%, практически не изменив значение точности для тренировочной выборки.

### 6. Если наблюдается улучшение метрик после фильтрации признаков хотя бы для одной из моделей, то для набора отфильтрованных признаков (пересечение множеств отфильтрованных признаков каждой модели или объединение множеств &ndash; не особо важно, главное описать, каким образом получен новый subset данных) заново построить наилучшие модели `KNeighborsClassifier`, `DecisionTreeClassifier`, `RandomForestClassifier`, сравнить модели в пункте 7 на одинаковом полученном наборе отфильтрованных признаков. Иначе &ndash; пропустить этот пункт.

Серьезное изменение метрик наблюдается только для модели k-ближайших соседей. Фильтрация признаков происходила по средствам анализа корреляции (`correlated_columns`) между признаками.

Построение лучших моделей с отфильтрованными признаками.

In [60]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

model_tree_params = {
    'max_depth': range(1, 11),
    'max_features': range(2, 12) #максимальное количество признаков, которые есть
}

model_tree = DecisionTreeClassifier(random_state=42)

best_model_tree_grid = GridSearchCV(
    model_tree, model_tree_params,
    cv = 5, 
    scoring='accuracy', # для меры оценки выбранной пары признаков
    n_jobs = -1,  # используем все ядра процессора
    verbose = True
)

best_model_tree_grid.fit(X_train_resampled[correlated_columns], y_train_resampled);

Fitting 5 folds for each of 100 candidates, totalling 500 fits


In [61]:
y_pred_tree = best_model_tree_grid.predict(X_test[correlated_columns])

In [62]:
accuracy_tree = accuracy_score(y_test, best_model_tree_grid.predict(X_test[correlated_columns]))
precision_tree = precision_score(y_test, best_model_tree_grid.predict(X_test[correlated_columns]),  average='weighted')
recall_tree = recall_score(y_test, best_model_tree_grid.predict(X_test[correlated_columns]),  average='weighted')
f_measure_tree = 2*(precision_tree*recall_tree)/(precision_tree+recall_tree)

In [63]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

model_forest_params = {
    'max_depth': range(1, 11),
    'max_features': range(2, 12) #максимальное количество признаков, которые есть
}

model_forest = RandomForestClassifier(random_state=42)

best_model_forest_grid = GridSearchCV(
    model_forest, model_forest_params,
    cv = 5, 
    scoring='accuracy', # для меры оценки выбранной пары признаков
    n_jobs = -1,  # используем все ядра процессора
    verbose = True
)

best_model_forest_grid.fit(X_train_resampled[correlated_columns], y_train_resampled);

Fitting 5 folds for each of 100 candidates, totalling 500 fits


In [64]:
y_pred_forest = best_model_forest_grid.predict(X_test[correlated_columns])

In [65]:
accuracy_forest = accuracy_score(y_test, best_model_forest_grid.predict(X_test[correlated_columns]))
precision_forest = precision_score(y_test, best_model_forest_grid.predict(X_test[correlated_columns]),  average='weighted')
recall_forest = recall_score(y_test, best_model_forest_grid.predict(X_test[correlated_columns]),  average='weighted')
f_measure_forest = 2*(precision_forest*recall_forest)/(precision_forest+recall_forest)

In [66]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

model_knn_params = {
    'n_neighbors': range(2, 16)
}

model_knn = KNeighborsClassifier()

best_model_knn_grid = GridSearchCV(
    model_knn, model_knn_params,
    cv = 5, 
    scoring='accuracy', # для меры оценки выбранной пары признаков
    n_jobs = -1,  # используем все ядра процессора
    verbose = True
)

best_model_knn_grid.fit(X_train_resampled[correlated_columns], y_train_resampled);

Fitting 5 folds for each of 14 candidates, totalling 70 fits


In [67]:
y_pred_knn = best_model_knn_grid.predict(X_test[correlated_columns])

In [68]:
accuracy_knn = accuracy_score(y_test, best_model_knn_grid.predict(X_test[correlated_columns]))
precision_knn = precision_score(y_test, best_model_knn_grid.predict(X_test[correlated_columns]),  average='weighted')
recall_knn = recall_score(y_test, best_model_knn_grid.predict(X_test[correlated_columns]),  average='weighted')
f_measure_knn = 2*(precision_knn*recall_knn)/(precision_knn+recall_knn)

### 7. Оценка качества построенных моделей:
   - Визуализировать любое полученное дерево решений.
     > Для вывода названий признаков в граф необходимо задать значение аргумента `feature_names` в `sklearn.tree.export_graphviz`, для вывода названий классов &ndash; `class_names` (перед кодированием целевого признака можно сохранить названия в отдельный массив).

In [72]:
import os
import graphviz 
from sklearn.tree import export_graphviz


def visualize_tree_with_names(model, feature_names, class_names, 
                             output_name='decision_tree', output_dir='./visualizations',
                             max_depth=None):

    os.makedirs(output_dir, exist_ok=True)
    
    dot_data = export_graphviz(
        model,
        feature_names=feature_names,  
        class_names=class_names,     
        filled=True,
        rounded=True,
        special_characters=True,
        max_depth=max_depth,         
        proportion=False,              # Показывать абсолютные значения
        impurity=True,                 # Показывать impurity
        node_ids=False,                # Не показывать ID узлов
        precision=2                    # Точность чисел
    )
    
    dot_path = os.path.join(output_dir, f'{output_name}.dot')
    with open(dot_path, 'w', encoding='utf-8') as f:
        f.write(dot_data)
    print(f"DOT файл сохранен: {dot_path}")
    

    try:
        graph = graphviz.Source(dot_data)
        png_path = os.path.join(output_dir, f'{output_name}.png')
        graph.render(filename=output_name, directory=output_dir, 
                    format='png', cleanup=True)
        print(f"PNG файл сохранен: {png_path}")
        return graph
        
    except Exception as e:
        print(f"Ошибка при конвертации: {e}")
        return None

best_model_tree = best_model_tree_grid.best_estimator_

graph = visualize_tree_with_names(
    model=best_model_tree,
    feature_names=list(correlated_columns),  
    class_names=["CNG", "Diesel", "Petrol"],       
    output_name='fuel_type',
    output_dir='.',
    max_depth=6
)

DOT файл сохранен: .\fuel_type.dot
PNG файл сохранен: .\fuel_type.png


<div align="center">
  <img src=".\fuel_type.png" width="1000px" title="Визуализация дерева решений"/>
  <p style="text-align: center">
    Рисунок 1 &ndash; Визуализация дерева решений
  </p>
</div>

Сравнить лучшие модели `KNeighborsClassifier`, `DecisionTreeClassifier`, `RandomForestClassifier` на **тестовой выборке**. Привести значения метрик $accuracy$, $precision$, $recall$, $\textit{f-measure}$.

In [70]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

print(f"DecisionTreeClassifier:")
print(f'Test accuracy: {accuracy_tree}')
print(f'Test precision: {precision_tree}')
print(f'Test recall: {recall_tree}')
print(f'Test f-measure: {f_measure_tree}')
print(f"  params:       {best_model_tree_grid.best_params_}")
print(f"  test score:   {accuracy_score(y_test, best_model_tree_grid.predict(X_test[correlated_columns]))}")

print(f"\n RandomForestClassifier:")
print(f'Test accuracy: {accuracy_forest}')
print(f'Test precision: {precision_forest}')
print(f'Test recall: {recall_forest}')
print(f'Test f-measure: {f_measure_forest}')
print(f"  params:       {best_model_forest_grid.best_params_}")
print(f"  test score:   {accuracy_score(y_test, best_model_forest_grid.predict(X_test[correlated_columns]))}")

print(f"\n KNeighborsClassifier:")
print(f'Test accuracy: {accuracy_knn}')
print(f'Test precision: {precision_knn}')
print(f'Test recall: {recall_knn}')
print(f'Test f-measure: {f_measure_knn}')
print(f"  params:       {best_model_knn_grid.best_params_}")
print(f"  test score:   {accuracy_score(y_test, best_model_knn_grid.predict(X_test[correlated_columns]))}")

DecisionTreeClassifier:
Test accuracy: 0.8630952380952381
Test precision: 0.9211846910771933
Test recall: 0.8630952380952381
Test f-measure: 0.8911943773796982
  params:       {'max_depth': 10, 'max_features': 5}
  test score:   0.8630952380952381

 RandomForestClassifier:
Test accuracy: 0.8928571428571429
Test precision: 0.9225079032743118
Test recall: 0.8928571428571429
Test f-measure: 0.907440376838667
  params:       {'max_depth': 10, 'max_features': 3}
  test score:   0.8928571428571429

 KNeighborsClassifier:
Test accuracy: 0.8616071428571429
Test precision: 0.9153820412673842
Test recall: 0.8616071428571429
Test f-measure: 0.8876809293442047
  params:       {'n_neighbors': 3}
  test score:   0.8616071428571429


На тестовой выборке наилучшую метрику accuracy имеет `RandomForestClassifier`. Это объясняется тем, что RandomForestClassifier представляет собой ансамблевый метод, основанный на построении множества деревьев решений. Каждое дерево обучается на случайной подвыборке данных (bootstrap sampling) и использует случайное подмножество признаков. Такой подход значительно снижает риск переобучения по сравнению с одиночными моделями, такими как KNN или деревья решений. В результате модель лучше обобщается на новых, невидимых данных, что и подтверждается высокой accuracy на тестовой выборке.